# CI-Triage-Env — GRPO Training

**Hardware target**: A10G Large (46 GB VRAM, 12 vCPU) — HuggingFace Space

**Set these as Space secrets** (Settings → Variables and secrets):
- `HF_TOKEN` — HuggingFace write token
- `HF_USERNAME` — your HF username
- `WANDB_API_KEY` — Weights & Biases key (get free at wandb.ai)

**Time budget**: SFT≈45 min + GRPO≈90 min = ~2.5 hours on A10G Large.

In [ ]:
# Cell 1 — Skip if using Docker image (deps already installed)
# Only run this if ci_triage_env is not importable
import importlib.util
if importlib.util.find_spec('ci_triage_env') is None:
    import subprocess
    subprocess.run('pip install -q -e /workspace', shell=True, check=True)
    print('Package installed.')
else:
    print('ci_triage_env already available — skipping install.')

In [ ]:
# Cell 2 — Authenticate
import os
from huggingface_hub import login

HF_TOKEN    = os.environ['HF_TOKEN']
HF_USERNAME = os.environ['HF_USERNAME']
WANDB_KEY   = os.environ.get('WANDB_API_KEY', '')

login(token=HF_TOKEN)

if WANDB_KEY:
    import wandb
    wandb.login(key=WANDB_KEY)
    os.environ['WANDB_PROJECT'] = 'ci-triage-env'
else:
    os.environ['WANDB_DISABLED'] = 'true'
    print('W&B disabled — set WANDB_API_KEY secret to enable')

SCENARIOS_REPO   = f'{HF_USERNAME}/ci-triage-scenarios'
SFT_DATASET_REPO = f'{HF_USERNAME}/ci-triage-sft'
MODEL_REPO       = f'{HF_USERNAME}/ci-triage-agent'
print(f'Authenticated as {HF_USERNAME}')

In [ ]:
# Cell 3 — Download scenarios from HF Hub and write as JSON files
# The dataset is stored as Parquet on HF Hub — we convert back to JSON
# so MockEnvClient and scenario_loader can read them normally.
from pathlib import Path
from datasets import load_dataset
import json

SCEN_DIR = Path('/data/scenarios')

existing = list(SCEN_DIR.rglob('*.json'))
if existing:
    print(f'Scenarios already present: {len(existing)} files — skipping')
else:
    total = 0
    for split in ['train', 'val', 'held_out']:
        try:
            ds = load_dataset(SCENARIOS_REPO, split=split, token=HF_TOKEN)
        except Exception as e:
            print(f'  {split}: skipped ({e})')
            continue

        split_dir = SCEN_DIR / split
        split_dir.mkdir(parents=True, exist_ok=True)

        for row in ds:
            if 'scenario_json' in row:
                payload = row['scenario_json']
            else:
                payload = json.dumps(dict(row))
            meta = json.loads(payload)
            sid = meta.get('scenario_id', f'unknown_{total}')
            (split_dir / f'{sid}.json').write_text(payload)
            total += 1

        print(f'  {split}: {len(ds)} scenarios written')

train_dir = SCEN_DIR / 'train'
print(f'Train scenarios ready: {len(list(train_dir.rglob("*.json")))}')

In [ ]:
# Cell 4 — Download SFT dataset from HF Hub
from pathlib import Path
from datasets import load_dataset, load_from_disk

SFT_DS_DIR = Path('/data/sft_dataset')

if SFT_DS_DIR.exists():
    ds = load_from_disk(str(SFT_DS_DIR))
    print(f'SFT dataset already present: {len(ds)} examples')
else:
    ds = load_dataset(SFT_DATASET_REPO, split='train', token=HF_TOKEN)
    SFT_DS_DIR.mkdir(parents=True, exist_ok=True)
    ds.save_to_disk(str(SFT_DS_DIR))
    print(f'Downloaded {len(ds)} SFT examples')

In [ ]:
# Cell 5 — SFT warmstart (~45 min on A10G Large)
from pathlib import Path
from ci_triage_env.training.sft import run_sft

SFT_CKPT = Path('/data/checkpoints/sft')

if SFT_CKPT.exists():
    print(f'SFT checkpoint found — skipping (delete /data/checkpoints/sft to retrain)')
else:
    run_sft(
        dataset_path=str(SFT_DS_DIR),
        output_dir=str(SFT_CKPT),
        num_epochs=2,
        per_device_batch_size=4,
        gradient_accumulation_steps=4,
    )
    print(f'SFT done → {SFT_CKPT}')

    from huggingface_hub import upload_folder
    upload_folder(
        folder_path=str(SFT_CKPT),
        repo_id=MODEL_REPO + '-sft',
        repo_type='model',
        token=HF_TOKEN,
        commit_message='SFT warmstart checkpoint (Qwen3.5-4B + LoRA)',
    )
    print(f'SFT checkpoint pushed to {MODEL_REPO}-sft')

In [ ]:
# Cell 6 — GRPO fine-tuning (~90 min for 100 steps on A10G Large)
from pathlib import Path
from ci_triage_env.training.mock_env_client import MockEnvClient
from ci_triage_env.training.grpo import run_grpo

SCEN_DIR   = Path('/data/scenarios')
SFT_CKPT   = Path('/data/checkpoints/sft')
GRPO_CKPT  = Path('/data/checkpoints/grpo')
GRPO_STEPS = 100

env_client = MockEnvClient(scenarios_dir=str(SCEN_DIR / 'train'))
print(f'MockEnvClient: {len(env_client.scenario_ids)} train scenarios')
print(f'Starting GRPO — {GRPO_STEPS} steps (~{GRPO_STEPS * 50 // 60} min)')

run_grpo(
    sft_checkpoint_dir=str(SFT_CKPT),
    output_dir=str(GRPO_CKPT),
    total_steps=GRPO_STEPS,
    env_client=env_client,
    scenarios_train_path=str(SCEN_DIR / 'train'),
    hyperparams={
        'per_device_train_batch_size': 1,
        'gradient_accumulation_steps': 4,
        'learning_rate': 5e-6,
        'kl_coef': 0.04,
        'num_generations': 4,
        'max_prompt_length': 2048,
        'max_completion_length': 256,
        'temperature': 0.8,
        'top_p': 0.95,
        'logging_steps': 5,
        'save_steps': 50,
        'report_to': 'wandb' if WANDB_KEY else 'none',
    },
)
print(f'GRPO done → {GRPO_CKPT}')

In [ ]:
# Cell 7 — Push final model to HF Hub
from huggingface_hub import upload_folder
from pathlib import Path

GRPO_CKPT = Path('/data/checkpoints/grpo')

upload_folder(
    folder_path=str(GRPO_CKPT),
    repo_id=MODEL_REPO,
    repo_type='model',
    token=HF_TOKEN,
    commit_message=f'GRPO-trained adapter — {GRPO_STEPS} steps on A10G Large',
)
print(f'Final model: https://huggingface.co/{MODEL_REPO}')